# 03 · CNNs in Depth `[local]`

**Goal:** understand convolutional neural networks well enough to build one from raw NumPy (forward *and* backward), reason about its shapes, parameters, compute cost and receptive field, and train a strong one on CIFAR-10.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Why fully-connected networks fail on images; weight sharing; the feature hierarchy |
| 📐 Math | Multi-channel convolution, output size, parameters, FLOPs, receptive field, backprop through conv, BatchNorm |
| 🔨 From scratch | Conv layer forward + backward (im2col) in NumPy, gradient-checked against PyTorch; parameter/FLOP/receptive-field calculators |
| 🚀 PyTorch | A CIFAR-10 CNN; filters and feature maps; equivariance; conv variants; BatchNorm train vs. eval |
| 📊 Evaluate | CNN vs. MLP (accuracy per parameter), per-class accuracy, confusions, confident mistakes |
| 🧪 Experiment | Remove BatchNorm, shuffle the pixels, go deeper without BatchNorm |

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.lib.stride_tricks import sliding_window_view
from torch.utils.flop_counter import FlopCounterMode
from cvlib.datasets import cifar10, CIFAR10_CLASSES
from cvlib.loaders import GPULoader, random_crop_flip, CIFAR_MEAN, CIFAR_STD
from cvlib import train as cvtrain
from cvlib.viz import plot_history

device = env.device
rng = np.random.default_rng(0)

---
# 🧠 1 · Intuition

### Why the MLP from module 02 doesn't scale to real images
A 32×32 color image has 3,072 numbers; a 224×224 photo has 150,528. A fully-connected layer with 1,000 neurons on that photo needs **150 million weights**, just for layer one. Worse, it treats every pixel as unrelated to its neighbors: a cat shifted 5 pixels to the right is, to an MLP, a completely different input.

### The two ideas that fix it
1. **Local connections:** each neuron looks only at a small patch (say 3×3), because nearby pixels are what matter for edges and textures.
2. **Weight sharing:** the *same* small filter slides over the whole image (the convolution from 01b). An edge detector that works in the top-left corner works everywhere.

Result: a 3×3 conv layer from 64 to 64 channels has ~37k weights **regardless of image size**, and if the input shifts, the output shifts the same way (**translation equivariance**).

### The feature hierarchy
Stack conv layers and each one sees a larger region of the original image (its **receptive field** grows):
- layer 1: edges and color blobs (like 01b's Sobel filters, but learned)
- middle layers: textures, corners, simple parts (fur, wheels, eyes)
- late layers: object parts and whole objects

**Pooling** (or stride 2) halves the resolution between stages, so later layers cover more of the image for the same cost.

---
# 📐 2 · The math

### Multi-channel convolution
Input $X$ has $C_{in}$ channels; the layer has $C_{out}$ filters, each of shape $C_{in} \times k \times k$ (every filter looks at **all** input channels):

$$Y[o, y, x] = b[o] + \sum_{c=1}^{C_{in}} \sum_{i=0}^{k-1}\sum_{j=0}^{k-1} W[o, c, i, j]\; X_{\text{padded}}[c,\; y\cdot s + i,\; x \cdot s + j]$$

### Output size
With input size $H$, kernel $k$, padding $p$, stride $s$:

$$H_{out} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

"Same" padding for odd $k$ and stride 1: $p = (k-1)/2$ keeps the size.

### Parameters and compute
$$\text{params} = C_{out} \cdot C_{in} \cdot k^2 \;(+\, C_{out} \text{ biases}) \qquad \text{MACs} = C_{out} \cdot C_{in} \cdot k^2 \cdot H_{out} \cdot W_{out}$$

A MAC is one multiply-add; **FLOPs ≈ 2 × MACs**. Parameters don't depend on image size; compute does.

### Receptive field
How many input pixels can influence one output pixel. Going layer by layer with kernel $k_l$ and stride $s_l$ (the "jump" $j$ is the distance in input pixels between neighboring outputs):

$$r_l = r_{l-1} + (k_l - 1)\, j_{l-1}, \qquad j_l = j_{l-1}\, s_l, \qquad r_0 = 1,\; j_0 = 1$$

### Backprop through a convolution
With upstream gradient $\partial Y$ (same shape as $Y$):
- $\partial W[o,c,i,j] = \sum_{n,y,x} \partial Y[n,o,y,x] \cdot X[n,c,y s + i, x s + j]$: correlate the **input** with the output gradient
- $\partial b[o] = \sum_{n,y,x} \partial Y[n,o,y,x]$
- $\partial X$: each input pixel collects gradient from every output position it contributed to. This is a convolution with the **flipped** kernel, also called a **transposed convolution**. (Same operation used for upsampling in segmentation, module 07.)

### Depthwise-separable convolution (MobileNet's trick)
Split a $k\times k$ conv into a **depthwise** conv (one $k \times k$ filter per channel, no mixing) + a **pointwise** 1×1 conv (mixing only). Cost ratio vs. a standard conv:

$$\frac{1}{C_{out}} + \frac{1}{k^2} \approx \frac{1}{9}\ \text{for } k = 3$$

### Batch normalization
For each channel, over the batch (and spatial positions):

$$\hat{x} = \frac{x - \mu_{\text{batch}}}{\sqrt{\sigma^2_{\text{batch}} + \epsilon}}, \qquad y = \gamma \hat{x} + \beta$$

$\gamma, \beta$ are learned. During training it uses the **current batch's** statistics and updates running averages. In `eval()` mode it uses the **running averages**. Forgetting `model.eval()` therefore changes predictions (section 4.6).

---
# 🔨 3 · From scratch: a conv layer in NumPy

### 3.1 · im2col: turning convolution into one matrix multiply

Looping over output pixels is slow. The standard trick: copy every $C_{in}\times k \times k$ input patch into a column of a big matrix (**im2col**). Then convolution is **one matrix multiply**: filters (rows) × patches (columns). GPUs are extremely good at matrix multiplies, which is why convs run fast.

In [ ]:
def im2col(x, k, stride=1, pad=0):
    """x: (N, C, H, W) → cols: (N, C*k*k, H_out*W_out)"""
    N, C, H, W = x.shape
    xp = np.pad(x, ((0, 0), (0, 0), (pad, pad), (pad, pad)))
    win = sliding_window_view(xp, (k, k), axis=(2, 3))[:, :, ::stride, ::stride]   # (N, C, Ho, Wo, k, k)
    Ho, Wo = win.shape[2], win.shape[3]
    cols = win.transpose(0, 1, 4, 5, 2, 3).reshape(N, C * k * k, Ho * Wo)
    return cols, Ho, Wo

class Conv2dNumpy:
    def __init__(self, c_in, c_out, k, stride=1, pad=0, seed=0):
        r = np.random.default_rng(seed)
        self.W = r.normal(0, np.sqrt(2 / (c_in * k * k)), (c_out, c_in, k, k))   # He init
        self.b = np.zeros(c_out)
        self.k, self.stride, self.pad = k, stride, pad

    def forward(self, x):
        self.x_shape = x.shape
        self.cols, Ho, Wo = im2col(x, self.k, self.stride, self.pad)
        out = self.W.reshape(len(self.W), -1) @ self.cols + self.b[:, None]      # (N, C_out, Ho*Wo)
        return out.reshape(x.shape[0], len(self.W), Ho, Wo)

    def backward(self, dout):
        N, C_out, Ho, Wo = dout.shape
        d = dout.reshape(N, C_out, Ho * Wo)
        self.dW = np.einsum("nol,nkl->ok", d, self.cols).reshape(self.W.shape)  # input patches × output grads
        self.db = d.sum(axis=(0, 2))
        dcols = np.einsum("ok,nol->nkl", self.W.reshape(C_out, -1), d)           # (N, C*k*k, Ho*Wo)
        # col2im: scatter every patch gradient back to where the patch came from (overlaps add up)
        _, C, H, W_ = self.x_shape
        k, s, p = self.k, self.stride, self.pad
        dxp = np.zeros((N, C, H + 2 * p, W_ + 2 * p))
        dcols = dcols.reshape(N, C, k, k, Ho, Wo)
        for i in range(k):
            for j in range(k):
                dxp[:, :, i:i + s * Ho:s, j:j + s * Wo:s] += dcols[:, :, i, j]
        return dxp[:, :, p:p + H, p:p + W_]

### 3.2 · Gradient check against PyTorch autograd

Same weights, same input, a random upstream gradient. Our forward and all three backward gradients should match PyTorch.

In [ ]:
for (k, stride, pad) in [(3, 1, 1), (5, 2, 2), (3, 2, 0)]:
    conv = Conv2dNumpy(c_in=3, c_out=4, k=k, stride=stride, pad=pad, seed=1)
    conv.b = rng.normal(size=4)
    x = rng.normal(size=(2, 3, 9, 9))
    out = conv.forward(x)
    g = rng.normal(size=out.shape)                       # pretend upstream gradient
    dx = conv.backward(g)

    tx = torch.tensor(x, requires_grad=True)
    tW = torch.tensor(conv.W, requires_grad=True)
    tb = torch.tensor(conv.b, requires_grad=True)
    tout = F.conv2d(tx, tW, tb, stride=stride, padding=pad)
    (tout * torch.tensor(g)).sum().backward()           # gives exactly the gradients for upstream g

    errs = [np.abs(a - b.detach().numpy()).max() for a, b in [(out, tout), (conv.dW, tW.grad), (conv.db, tb.grad), (dx, tx.grad)]]
    print(f"k={k} stride={stride} pad={pad} | out shape {out.shape} | max error: forward {errs[0]:.1e}, dW {errs[1]:.1e}, db {errs[2]:.1e}, dX {errs[3]:.1e}")

All errors at float64 rounding level (~1e-15): your conv layer's forward and backward passes are exactly what PyTorch computes.

### 3.3 · Calculators: output size, parameters, FLOPs, receptive field

Being able to compute these in your head (or quickly in code) is essential for designing and debugging networks, and a common interview topic.

In [ ]:
def conv_out(h, k, s=1, p=0, d=1):
    return (h + 2 * p - d * (k - 1) - 1) // s + 1           # d = dilation (k_eff = d*(k-1)+1)

def describe(layers, h=32, c=3):
    """layers: list of ("conv", c_out, k, s, p) or ("pool", k, s). Prints shapes, params, MACs, receptive field."""
    r, j, total_p, total_m = 1, 1, 0, 0
    print(f"{'layer':<18}{'output':>14}{'params':>10}{'MACs':>13}{'RF':>6}")
    for L in layers:
        if L[0] == "conv":
            _, co, k, s, p = L
            h2 = conv_out(h, k, s, p)
            params = co * c * k * k + co
            macs = co * c * k * k * h2 * h2
            c = co
        else:
            _, k, s = L
            h2, params, macs = conv_out(h, k, s), 0, 0
        r, j = r + (k - 1) * j, j * s
        total_p += params; total_m += macs; h = h2
        print(f"{str(L):<18}{f'{c}×{h}×{h}':>14}{params:>10,}{macs:>13,}{r:>6}")
    print(f"{'total':<18}{'':>14}{total_p:>10,}{total_m:>13,}")
    return total_p, total_m

arch = [("conv", 32, 3, 1, 1), ("conv", 32, 3, 1, 1), ("pool", 2, 2),
        ("conv", 64, 3, 1, 1), ("conv", 64, 3, 1, 1), ("pool", 2, 2),
        ("conv", 128, 3, 1, 1), ("conv", 128, 3, 1, 1), ("pool", 2, 2)]
my_params, my_macs = describe(arch)

Each output of the last conv layer has a receptive field of **32 pixels** (36 after the final pooling): the full 32×32 image, so each final feature can "see" the whole object. We'll build exactly this network in PyTorch next and check these numbers against PyTorch's own counters.

---
# 🚀 4 · The PyTorch way: a CIFAR-10 CNN

### 4.1 · Data

CIFAR-10: 60,000 tiny 32×32 color photos in 10 classes. Far harder than MNIST: real backgrounds, poses, lighting. We hold out 5,000 training images for validation.

In [ ]:
X_all, y_all = cifar10("train")
X_test, y_test = cifar10("test")
perm = rng.permutation(len(X_all))
X_train, y_train = X_all[perm[:45_000]], y_all[perm[:45_000]]
X_val, y_val = X_all[perm[45_000:]], y_all[perm[45_000:]]
print(f"train {X_train.shape} | val {X_val.shape} | test {X_test.shape} | dtype {X_train.dtype}")

fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for ax, i in zip(axes.flat, rng.choice(len(X_train), 20, replace=False)):
    ax.imshow(X_train[i]); ax.set_title(CIFAR10_CLASSES[y_train[i]], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

`GPULoader` (in `cvlib/loaders.py`) keeps the uint8 images on the device and converts, augments and normalizes each batch there, so loading never slows down training. The augmentation (random crop + horizontal flip) is a preview of module 05.

In [ ]:
train_loader = GPULoader(X_train, y_train, batch_size=128, shuffle=True, augment=random_crop_flip, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)
val_loader = GPULoader(X_val, y_val, batch_size=1000, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)
test_loader = GPULoader(X_test, y_test, batch_size=1000, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)
xb, yb = next(iter(train_loader))
print("batch:", tuple(xb.shape), xb.dtype, xb.device, f"| mean {xb.mean():.2f} std {xb.std():.2f}")

### 4.2 · The model

Conv → BatchNorm → ReLU is the standard building block. Convs before BatchNorm use `bias=False`, because BN's $\beta$ already provides a bias. **Global average pooling** at the end averages each final feature map to one number, so the classifier head is tiny.

In [ ]:
def conv_bn(c_in, c_out, bn=True):
    layers = [nn.Conv2d(c_in, c_out, 3, padding=1, bias=not bn)]
    if bn:
        layers.append(nn.BatchNorm2d(c_out))
    layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)

class SimpleCNN(nn.Module):
    def __init__(self, widths=(32, 64, 128), bn=True, n_classes=10):
        super().__init__()
        stages, c = [], 3
        for w in widths:
            stages += [conv_bn(c, w, bn), conv_bn(w, w, bn), nn.MaxPool2d(2)]
            c = w
        self.features = nn.Sequential(*stages)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, n_classes))

    def forward(self, x):
        return self.head(self.features(x))

model = SimpleCNN()
n_params = sum(p.numel() for p in model.parameters())
with FlopCounterMode(display=False) as fc:
    model(torch.zeros(1, 3, 32, 32))
print(f"PyTorch: {n_params:,} parameters | {fc.get_total_flops() / 2:,.0f} MACs per image")
print(f"ours   : {my_params:,} conv parameters | {my_macs:,} conv MACs (no BN/head: compare the conv parts)")

The parameter counts differ slightly because our calculator includes conv biases while the PyTorch model uses BatchNorm parameters plus a linear head instead. The MACs agree to within the small linear head: the conv layers are where all the compute is.

### 4.3 · Train

Adam with a **cosine learning-rate schedule**: start at 2e-3 and smoothly decay toward 0, so the final epochs make small, careful steps. (Schedules are covered properly in module 05.)

In [ ]:
torch.manual_seed(0)
model = SimpleCNN().to(device)
EPOCHS = 12
opt = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=5e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
t0 = time.perf_counter()
history = cvtrain.fit(model, train_loader, val_loader, nn.CrossEntropyLoss(), opt, epochs=EPOCHS, device=device, scheduler=sched)
print(f"total training time: {time.perf_counter() - t0:.0f}s on {device}")
trained_state = {k: v.clone() for k, v in model.state_dict().items()}   # a safe copy of the trained weights
plot_history(history)

### 4.4 · What did it learn? First-layer filters and feature maps

In [ ]:
w1 = model.features[0][0].weight.detach().cpu()                 # (32, 3, 3, 3)
w1 = (w1 - w1.min()) / (w1.max() - w1.min())
fig, axes = plt.subplots(2, 16, figsize=(13, 2))
for ax, f in zip(axes.flat, w1):
    ax.imshow(f.permute(1, 2, 0)); ax.axis("off")
plt.suptitle("All 32 first-layer filters (3×3×RGB)"); plt.show()

At 3×3 pixels, filters are hard to interpret by eye: each is a tiny color-and-contrast pattern (e.g. "brighter on the left", "green above red"). Larger first-layer filters, like the 7×7 ones of the ImageNet-trained ResNet-50 in module 04, clearly show oriented edges and color blobs. Small filters build up the same detectors over several layers.

In [ ]:
img_idx = int(np.nonzero(y_test == CIFAR10_CLASSES.index("horse"))[0][3])
x_one = next(iter(GPULoader(X_test[img_idx:img_idx + 1], y_test[img_idx:img_idx + 1], mean=CIFAR_MEAN, std=CIFAR_STD, device=device)))[0]
acts, h = [], x_one
model.eval()
with torch.no_grad():
    for i, layer in enumerate(model.features):
        h = layer(h)
        if isinstance(layer, nn.MaxPool2d):
            acts.append(h[0].cpu())
fig, axes = plt.subplots(3, 9, figsize=(13, 4.6))
for row, a in enumerate(acts):
    axes[row, 0].imshow(X_test[img_idx]); axes[row, 0].set_title("input", fontsize=8)
    for col in range(1, 9):
        axes[row, col].imshow(a[col - 1], cmap="viridis")
    axes[row, 1].set_title(f"stage {row + 1}: {a.shape[1]}×{a.shape[2]} maps, {a.shape[0]} channels", fontsize=8, loc="left")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

Stage 1 maps (16×16) respond to small local patterns (edges, color changes) scattered across the image. By stage 3 (4×4), each map is a coarse "is feature X present in this region?" signal. Less spatial detail, more meaning: that's the feature hierarchy.

### 4.5 · Translation equivariance, demonstrated

Shift the input 8 pixels right. The conv feature maps should shift the same way. Global average pooling then makes the final prediction (nearly) **invariant** to the shift.

In [ ]:
x_shift = torch.roll(x_one, shifts=8, dims=3)
with torch.no_grad():
    f1 = model.features[0](x_one)[0, 5].cpu()                      # channel 5 of layer 1
    f2 = model.features[0](x_shift)[0, 5].cpu()
    p1 = F.softmax(model(x_one), 1)[0].cpu(); p2 = F.softmax(model(x_shift), 1)[0].cpu()
diff = (torch.roll(f1, 8, dims=1) - f2).abs()[:, 9:-1].max()      # ignore the wrapped-around border columns
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, im, t in zip(axes, [f1, f2, torch.roll(f1, 8, dims=1)], ["features of original", "features of shifted input", "original features, shifted"]):
    ax.imshow(im, cmap="viridis"); ax.set_title(t, fontsize=9); ax.axis("off")
plt.show()
print(f"max difference (away from borders): {diff:.2e} → shifting the input just shifts the feature map")
print(f"prediction: original '{CIFAR10_CLASSES[p1.argmax()]}' ({p1.max():.0%}) | shifted '{CIFAR10_CLASSES[p2.argmax()]}' ({p2.max():.0%})")

### 4.6 · Conv variants: same job, very different cost

Measured for a layer mapping 128 → 128 channels on a 32×32 input.

| Variant | What it does | Used in |
|---|---|---|
| standard 3×3 | mixes space and channels at once | everywhere |
| **1×1** | mixes channels only, at every pixel | bottlenecks (ResNet), channel projection |
| **dilated 3×3** (d=2) | 3×3 with gaps: 5×5 reach for 3×3 cost | segmentation (DeepLab, module 07) |
| **grouped** (g=4) | 4 independent convs on channel groups | ResNeXt |
| **depthwise-separable** | per-channel 3×3, then 1×1 mixing | MobileNet, EfficientNet (module 04) |

In [ ]:
C, HW = 128, 32
variants = {
    "standard 3×3": nn.Conv2d(C, C, 3, padding=1, bias=False),
    "1×1": nn.Conv2d(C, C, 1, bias=False),
    "dilated 3×3 (d=2)": nn.Conv2d(C, C, 3, padding=2, dilation=2, bias=False),
    "grouped 3×3 (g=4)": nn.Conv2d(C, C, 3, padding=1, groups=4, bias=False),
    "depthwise-separable": nn.Sequential(nn.Conv2d(C, C, 3, padding=1, groups=C, bias=False), nn.Conv2d(C, C, 1, bias=False)),
}
x_v = torch.randn(32, C, HW, HW, device=device)
print(f"{'variant':<22}{'params':>9}{'MACs/image':>14}{'reach':>7}{'time (ms, batch 32)':>22}")
for name, m in variants.items():
    m = m.to(device)
    with FlopCounterMode(display=False) as fc:
        m(torch.zeros(1, C, HW, HW, device=device))
    with torch.no_grad():
        for _ in range(3): m(x_v)                                      # warm-up
        if device == "mps": torch.mps.synchronize()
        t0 = time.perf_counter()
        for _ in range(20): m(x_v)
        if device == "mps": torch.mps.synchronize()
    ms = (time.perf_counter() - t0) / 20 * 1000
    reach = {"1×1": 1, "dilated 3×3 (d=2)": 5}.get(name, 3)
    print(f"{name:<22}{sum(p.numel() for p in m.parameters()):>9,}{fc.get_total_flops() / 2:>14,.0f}{reach:>7}{ms:>22.2f}")

Depthwise-separable uses **~9× fewer** parameters and MACs than a standard conv, but it is usually **not** 9× faster: tiny per-channel convs use the hardware poorly. FLOPs ≠ speed, a theme of module 04 and 15.

### 4.7 · BatchNorm: train mode vs. eval mode

The classic bug: predicting in `train()` mode. BatchNorm then normalizes with the statistics of the *current batch*, so a prediction depends on which other images are in the batch, and a batch of one image is normalized against itself.

In [ ]:
x8 = next(iter(test_loader))[0][:8]
model.eval()
with torch.no_grad():
    p_eval = model(x8).argmax(1)
model.train()
with torch.no_grad():                                               # no_grad does NOT switch BN to eval mode!
    p_train_batch = model(x8).argmax(1)
    p_train_single = torch.cat([model(x8[i:i + 1]).argmax(1) for i in range(8)])
model.eval()
print("eval mode           :", [CIFAR10_CLASSES[i][:5] for i in p_eval.tolist()])
print("train mode, batch 8 :", [CIFAR10_CLASSES[i][:5] for i in p_train_batch.tolist()])
print("train mode, 1 image :", [CIFAR10_CLASSES[i][:5] for i in p_train_single.tolist()])
model.load_state_dict(trained_state);      # train-mode passes also changed BN's running statistics: restore them

---
# 📊 5 · Evaluate

### 5.1 · Test accuracy, and CNN vs. MLP

Train a module-02-style MLP on the same data with the same budget and compare accuracy *per parameter*.

In [ ]:
_, cnn_test_acc = cvtrain.evaluate(model, test_loader, nn.CrossEntropyLoss(), device)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(3072, 512), nn.ReLU(), nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 10))
    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
mlp = MLP().to(device)
opt_m = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=5e-4)
h_mlp = cvtrain.fit(mlp, train_loader, val_loader, nn.CrossEntropyLoss(), opt_m, epochs=EPOCHS, device=device,
                    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt_m, EPOCHS), verbose=False)
_, mlp_test_acc = cvtrain.evaluate(mlp, test_loader, nn.CrossEntropyLoss(), device)

print(f"{'model':<6}{'params':>10}{'test acc':>10}")
print(f"{'MLP':<6}{sum(p.numel() for p in mlp.parameters()):>10,}{mlp_test_acc:>10.4f}")
print(f"{'CNN':<6}{sum(p.numel() for p in model.parameters()):>10,}{cnn_test_acc:>10.4f}")

The CNN is far more accurate with **~6× fewer parameters**. That's the payoff of building the right assumptions (locality, weight sharing) into the architecture: its **inductive bias**.

### 5.2 · Per-class accuracy and the most common confusions

In [ ]:
logits, labels = cvtrain.predict(model, test_loader, device)
preds = logits.argmax(1)
cm = torch.zeros(10, 10, dtype=torch.long)
cm.index_put_((labels.cpu(), preds), torch.ones_like(preds), accumulate=True)
per_class = (cm.diagonal() / cm.sum(1)).numpy()
order = np.argsort(per_class)
plt.figure(figsize=(8, 3))
plt.barh([CIFAR10_CLASSES[i] for i in order], per_class[order]); plt.xlim(0.5, 1); plt.grid(alpha=0.3, axis="x")
plt.title("Per-class test accuracy"); plt.tight_layout(); plt.show()

off = cm.clone().fill_diagonal_(0)
top = torch.topk(off.flatten(), 5)
for v, idx in zip(top.values, top.indices):
    i, j = divmod(int(idx), 10)
    print(f"{CIFAR10_CLASSES[i]:>10} → {CIFAR10_CLASSES[j]:<10} {int(v)} times")

Cat ↔ dog is the classic hardest pair: similar shapes, fur and poses at 32×32. The animal classes are generally harder than the vehicles.

### 5.3 · Most confident mistakes

In [ ]:
probs = F.softmax(logits, 1)
wrong = torch.nonzero(preds != labels.cpu()).ravel()
conf = probs[wrong, preds[wrong]]
show_idx = wrong[torch.argsort(conf, descending=True)[:12]]
fig, axes = plt.subplots(2, 6, figsize=(12, 4.6))
for ax, i in zip(axes.flat, show_idx.tolist()):
    ax.imshow(X_test[i]); ax.axis("off")
    ax.set_title(f"true {CIFAR10_CLASSES[y_test[i]]}\npred {CIFAR10_CLASSES[preds[i]]} ({probs[i, preds[i]]:.0%})", fontsize=8)
plt.suptitle(f"{len(wrong)} test errors: the most confident ones"); plt.tight_layout(); plt.show()

Some are honest ambiguities at 32×32 resolution, and a few look mislabeled. But notice the confidence: the network is often **very sure when wrong**. Modern networks tend to be *overconfident*; module 14 measures and fixes this (calibration).

---
# 🧪 6 · Experiments: predict first, then run

To stay quick, each experiment trains for 4 epochs on 20,000 images.

In [ ]:
X_small, y_small = X_train[:20_000], y_train[:20_000]

def quick_run(make_model, X=X_small, y=y_small, Xv=X_val, epochs=4, lr=2e-3):
    torch.manual_seed(0)
    m = make_model().to(device)
    tl = GPULoader(X, y, batch_size=128, shuffle=True, augment=random_crop_flip, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)
    vl = GPULoader(Xv, y_val, batch_size=1000, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)
    o = torch.optim.Adam(m.parameters(), lr=lr)
    return cvtrain.fit(m, tl, vl, nn.CrossEntropyLoss(), o, epochs=epochs, device=device, verbose=False)

**Experiment 1: remove BatchNorm.** Predict how the first few epochs change.

In [ ]:
# Prediction: ...
for bn in [True, False]:
    h = quick_run(lambda: SimpleCNN(bn=bn))
    print(f"BatchNorm={bn!s:<5}: val acc per epoch {[round(a, 3) for a in h['val_acc']]}")
# BatchNorm keeps every layer's inputs well-scaled, so the network trains faster and tolerates higher learning rates.

**Experiment 2: shuffle the pixels.** Apply one fixed random permutation to the pixel positions of *every* image (train, val, test alike). No information is lost; a network could in principle learn the shuffled task. Predict what happens to the MLP and to the CNN.

In [ ]:
# Prediction: ...
pix = rng.permutation(32 * 32)
shuffle_px = lambda X: X.reshape(len(X), 32 * 32, 3)[:, pix].reshape(X.shape)
show_n = 6
fig, axes = plt.subplots(2, show_n, figsize=(10, 3.4))
for k in range(show_n):
    axes[0, k].imshow(X_small[k]); axes[1, k].imshow(shuffle_px(X_small[k:k + 1])[0])
    axes[0, k].axis("off"); axes[1, k].axis("off")
plt.suptitle("top: normal | bottom: same images, pixels shuffled (same shuffle for all)"); plt.show()

Xs_small, Xs_val = shuffle_px(X_small), shuffle_px(X_val)
for name, mk in [("MLP", MLP), ("CNN", SimpleCNN)]:
    torch.manual_seed(0)
    res = []
    for Xtr_, Xva_ in [(X_small, X_val), (Xs_small, Xs_val)]:
        m = mk().to(device)
        tl = GPULoader(Xtr_, y_small, batch_size=128, shuffle=True, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)   # no crop/flip: they'd break the shuffle
        vl = GPULoader(Xva_, y_val, batch_size=1000, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)
        h = cvtrain.fit(m, tl, vl, nn.CrossEntropyLoss(), torch.optim.Adam(m.parameters(), 1e-3), epochs=4, device=device, verbose=False)
        res.append(h["val_acc"][-1])
    print(f"{name}: normal {res[0]:.3f} | shuffled {res[1]:.3f}")

The MLP barely notices: it never used pixel neighborhoods anyway (any fixed shuffle is just a relabeling of its inputs). The CNN collapses, even **below** the MLP, because its whole advantage is the assumption that **nearby pixels belong together**. On shuffled images that assumption is wrong, and small 3×3 filters can't reach the pixels that actually belong together. This is *inductive bias* in one experiment, and it will come back when ViTs (module 08) relax this assumption.

**Experiment 3: deeper without BatchNorm.** A plain stack of 12 conv layers with no BatchNorm vs. the same with BatchNorm. Predict which learns at all.

In [ ]:
# Prediction: ...
class DeepPlain(nn.Module):
    def __init__(self, depth=12, bn=False):
        super().__init__()
        layers, c = [conv_bn(3, 64, bn)], 64
        for i in range(depth - 1):
            layers.append(conv_bn(c, c, bn))
            if i in (3, 7):
                layers.append(nn.MaxPool2d(2))
        self.f = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 10))
        for m in self.modules():                                     # deliberately small init, as in early deep-learning days
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, 0, 0.01)
    def forward(self, x):
        return self.head(self.f(x))

for bn in [False, True]:
    h = quick_run(lambda: DeepPlain(bn=bn), epochs=3)
    print(f"12 conv layers, BatchNorm={bn!s:<5}: val acc per epoch {[round(a, 3) for a in h['val_acc']]}")
# Without BN, the small weights shrink the signal at every layer, and the gradients with it,
# so learning starts slowly and crawls. BatchNorm re-scales activations at every layer and trains far faster.
# ResNet's skip connections (module 04) are the other big fix.

---
# 📝 7 · Recap

**Cheat sheet**
- CNN = **local connections + weight sharing** → far fewer parameters, translation equivariance.
- Output size $= \lfloor (H + 2p - k)/s \rfloor + 1$. Params $= C_{out} C_{in} k^2$. MACs = params × output pixels. FLOPs ≈ 2 × MACs.
- Receptive field grows with depth and stride: $r_l = r_{l-1} + (k_l - 1) j_{l-1}$.
- Backprop: $\partial W$ = input patches × output gradient; $\partial X$ = transposed convolution. **im2col** turns conv into one matrix multiply.
- 1×1 mixes channels; dilated widens reach cheaply; depthwise-separable is ~9× cheaper on paper, but FLOPs ≠ speed.
- **Conv → BN → ReLU**; BN uses batch stats in `train()`, running stats in `eval()`. Always `model.eval()` for inference.
- Global average pooling makes the head tiny and the prediction shift-tolerant.
- Look at per-class accuracy, the confusion pairs, and confident mistakes, not just overall accuracy.

**Interview-style questions**
1. Why do CNNs need far fewer parameters than MLPs on images? What two assumptions make that possible?
2. Compute the output shape, parameters and MACs of `Conv2d(64, 128, 3, stride=2, padding=1)` on a 56×56 input.
3. What is the receptive field after three 3×3 convs with stride 1? With a 2×2 pool after each?
4. What is the gradient of a convolution with respect to its input? Where else does that operation appear?
5. Why is depthwise-separable convolution cheaper, and why isn't it proportionally faster?
6. What does BatchNorm do differently in training and evaluation, and what bug does that cause?
7. Equivariance vs. invariance: which do conv layers have, and how does a CNN classifier become (approximately) shift-invariant?

**Next:** `04_cnn_architectures`, the famous architectures that took these blocks from 85% on CIFAR to superhuman ImageNet accuracy.